# Time Series com MongoDB — via mongosh

Este notebook explora **Time Series Collections** do MongoDB usando o **mongosh** (MongoDB Shell) diretamente.
Todos os comandos podem também ser executados em um terminal com `mongosh` aberto.

## O que você vai aprender

| Seção | Conteúdo |
|---|---|
| 1 | Setup: iniciar o MongoDB e verificar a conexão |
| 2 | Básico: criar coleção TS, inserir e consultar documentos |
| 3 | Filtros e agregações: `$group`, downsampling, alertas |
| 4 | Janelas temporais: `$setWindowFields`, média móvel, delta, percentis |
| 5 | Caso real: pipeline completo para sensores IoT |
| 6 | Performance: índices, `explain()` e TTL automático |

> **Pré-requisito:** MongoDB 8.3 instalado via `postBuild`. Execute a Seção 1 antes de prosseguir.

---

## Seção 1 — Setup

### 1.2 Verificar a conexão

O retorno `{ ok: 1 }` confirma que o servidor está respondendo.

In [1]:
! mongosh --quiet --eval 'db.adminCommand({ ping: 1 })' admin

]0;mongosh mongodb://127.0.0.1:27017/admin?directConnection=true&serverSelectionTimeoutMS=2000{ ok: 1 }
]0;

---
## Seção 2 — Operações Básicas

### 2.1 Criar uma Time Series Collection

Time Series Collections exigem três parâmetros na criação:

| Parâmetro | Descrição |
|---|---|
| `timeField` | Campo com o timestamp — deve conter valores `Date` |
| `metaField` | Campo que identifica a série (ex: id do sensor) |
| `granularity` | Resolução dos dados: `seconds`, `minutes` ou `hours` |

O MongoDB usa esses parâmetros para otimizar o armazenamento interno em **buckets** (compressão por coluna),
sem alterar a API de consulta.

In [30]:
!mongosh --quiet --eval "show dbs"

MongoshInvalidInputError: [COMMON-10001] Invalid URI: show dbs


In [16]:
!mongosh --quiet --eval "db.getCollectionNames().join('\n');" admin

]0;mongosh mongodb://127.0.0.1:27017/admin?directConnection=true&serverSelectionTimeoutMS=2000system.version
]0;

In [ ]:
! mongosh --quiet --eval 'db.adminCommand({ ping: 1 })' admin

In [20]:
%%bash
mongosh --quiet << 'EOF'
use timeseries_lab

db.dropDatabase()

db.createCollection('temperatura', {
  timeseries: {
    timeField:   'timestamp',
    metaField:   'sensor_id',
    granularity: 'seconds'
  }
})

show collections
EOF

test> switched to db timeseries_lab
timeseries_lab> { ok: 1, dropped: 'timeseries_lab' }
timeseries_lab> 
timeseries_lab> | | | | | | { ok: 1 }
timeseries_lab> 
timeseries_lab> temperatura                 [time-series]
system.buckets.temperatura
system.views
timeseries_lab> 

### 2.2 Confirmar o tipo da collection

`getCollectionInfos` retorna os metadados da collection, incluindo o tipo `timeseries`
e os parâmetros de configuração.

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
db.getCollectionInfos({ name: 'temperatura' })
EOF

### 2.3 Inserir um único documento

O campo `timeField` deve conter um objeto `Date`. Use `new Date('...')` no formato ISO 8601 UTC.

```
// No terminal mongosh:
use timeseries_lab
db.temperatura.insertOne({ timestamp: new Date(), sensor_id: 'sensor-A1', temperatura_c: 22.5 })
```

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
db.temperatura.insertOne({
  timestamp:     new Date('2024-01-15T10:00:00Z'),
  sensor_id:     'sensor-A1',
  temperatura_c: 22.5,
  umidade_pct:   65.0,
  localizacao:   'sala-servidores'
})
EOF

### 2.4 Inserir múltiplos documentos com um loop JavaScript

Geramos **3 horas** de leituras a cada **1 minuto** para 3 sensores = **540 documentos**.
`insertMany` é sempre preferível a chamadas individuais em loop para melhor desempenho.

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
var docs    = [];
var start   = new Date('2024-01-15T08:00:00Z');
var sensores = ['sensor-A1', 'sensor-A2', 'sensor-B1'];

for (var m = 0; m < 180; m++) {
  var ts = new Date(start.getTime() + m * 60000);
  sensores.forEach(function(sensor) {
    docs.push({
      timestamp:     ts,
      sensor_id:     sensor,
      temperatura_c: Math.round((20 + Math.random() * 7) * 100) / 100,
      umidade_pct:   Math.round((55 + Math.random() * 20) * 100) / 100,
      localizacao:   sensor.startsWith('sensor-A') ? 'sala-servidores' : 'datacenter'
    });
  });
}

db.temperatura.insertMany(docs);
print('Documentos inseridos: ' + docs.length);
EOF

### 2.5 Consultas simples com `find()`

A API de consulta é **idêntica** a qualquer collection MongoDB — sem aprendizado adicional.

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
// 5 leituras mais recentes do sensor-A1
db.temperatura
  .find(
    { sensor_id: 'sensor-A1' },
    { _id: 0, timestamp: 1, temperatura_c: 1, umidade_pct: 1 }
  )
  .sort({ timestamp: -1 })
  .limit(5)
EOF

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
// Total de documentos por sensor
['sensor-A1', 'sensor-A2', 'sensor-B1'].forEach(function(s) {
  print(s + ': ' + db.temperatura.countDocuments({ sensor_id: s }) + ' leituras');
});
EOF

---
## Seção 3 — Filtros e Agregações

### 3.1 Filtrar por intervalo de tempo

`$gte` e `$lt` no `timeField` é a base de qualquer query em séries temporais.
O MongoDB utiliza o índice de tempo **automaticamente** — nenhuma configuração adicional necessária.

```
// No terminal:
db.temperatura.find({ timestamp: { $gte: new Date('2024-01-15T09:00:00Z'), $lt: new Date('2024-01-15T10:00:00Z') } })
```

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
db.temperatura.find(
  {
    timestamp: {
      $gte: new Date('2024-01-15T09:00:00Z'),
      $lt:  new Date('2024-01-15T10:00:00Z')
    },
    sensor_id: 'sensor-A1'
  },
  { _id: 0, timestamp: 1, temperatura_c: 1 }
).sort({ timestamp: 1 }).limit(5)
EOF

### 3.2 Estatísticas por sensor com `$group`

Calcula temperatura média, máxima e mínima, além de umidade média e total de leituras para cada sensor.

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
db.temperatura.aggregate([
  {
    $group: {
      _id:            '$sensor_id',
      temp_media:     { $avg: '$temperatura_c' },
      temp_maxima:    { $max: '$temperatura_c' },
      temp_minima:    { $min: '$temperatura_c' },
      umidade_media:  { $avg: '$umidade_pct'   },
      total_leituras: { $sum: 1 }
    }
  },
  { $sort: { _id: 1 } }
])
EOF

### 3.3 Downsampling — Agrupar por hora com `$dateTrunc`

Downsampling reduz a resolução temporal, trocando leituras individuais por resumos por período.
Ideal para dashboards de longo prazo ou arquivamento.
`$dateTrunc` trunca o timestamp para a unidade desejada (`hour`, `minute`, `day`, etc.).

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
db.temperatura.aggregate([
  {
    $group: {
      _id: {
        hora:   { $dateTrunc: { date: '$timestamp', unit: 'hour' } },
        sensor: '$sensor_id'
      },
      temp_media: { $avg: '$temperatura_c' },
      temp_max:   { $max: '$temperatura_c' },
      temp_min:   { $min: '$temperatura_c' },
      leituras:   { $sum: 1 }
    }
  },
  { $sort: { '_id.hora': 1, '_id.sensor': 1 } }
])
EOF

### 3.4 Detectar leituras acima de limiar (alertas)

Combina `$match` com condições compostas (`$or`) e `$addFields` para identificar e classificar eventos de alerta.

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
db.temperatura.aggregate([
  {
    $match: {
      $or: [
        { temperatura_c: { $gt: 24.0 } },
        { umidade_pct:   { $gt: 68.0 } }
      ]
    }
  },
  {
    $addFields: {
      alerta_temp:    { $gt: ['$temperatura_c', 24.0] },
      alerta_umidade: { $gt: ['$umidade_pct',   68.0] }
    }
  },
  { $sort:  { timestamp: 1 } },
  { $limit: 8 },
  { $project: { _id: 0, timestamp: 1, sensor_id: 1,
                temperatura_c: 1, umidade_pct: 1,
                alerta_temp: 1, alerta_umidade: 1 } }
])
EOF

---
## Seção 4 — Janelas Temporais com `$setWindowFields`

`$setWindowFields` (MongoDB 5.0+) calcula estatísticas dentro de **janelas deslizantes** sem múltiplas queries.
É o equivalente às **window functions** do SQL (`OVER PARTITION BY ... ORDER BY ...`).

```
$setWindowFields: {
  partitionBy: <campo de agrupamento>,   // equivale ao PARTITION BY
  sortBy:      { campo: 1 },             // equivale ao ORDER BY
  output: {
    novo_campo: { $operador: ..., window: { ... } }
  }
}
```

### 4.1 Média Móvel (Moving Average)

Suaviza ruídos calculando a média das últimas N leituras para cada ponto da série.
`window: { documents: [-4, 0] }` define uma janela de **5 pontos** (atual + 4 anteriores).

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
db.temperatura.aggregate([
  { $match: { sensor_id: 'sensor-A1' } },
  { $sort:  { timestamp: 1 } },
  {
    $setWindowFields: {
      partitionBy: '$sensor_id',
      sortBy:      { timestamp: 1 },
      output: {
        media_movel_5: {
          $avg:   '$temperatura_c',
          window: { documents: [-4, 0] }
        }
      }
    }
  },
  { $limit: 10 },
  { $project: { _id: 0, timestamp: 1, temperatura_c: 1, media_movel_5: 1 } }
])
EOF

### 4.2 Soma Acumulada (Cumulative Sum)

Acumula valores do primeiro documento até o atual.
`'unbounded'` expande a janela desde o início da partição, independente do número de documentos.

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
db.temperatura.aggregate([
  { $match: { sensor_id: 'sensor-A1' } },
  { $sort:  { timestamp: 1 } },
  {
    $setWindowFields: {
      partitionBy: '$sensor_id',
      sortBy:      { timestamp: 1 },
      output: {
        soma_acumulada: {
          $sum:   '$temperatura_c',
          window: { documents: ['unbounded', 'current'] }
        },
        numero_leitura: {
          $sum:   1,
          window: { documents: ['unbounded', 'current'] }
        }
      }
    }
  },
  { $limit: 8 },
  { $project: { _id: 0, timestamp: 1, temperatura_c: 1,
                soma_acumulada: 1, numero_leitura: 1 } }
])
EOF

### 4.3 Delta entre leituras consecutivas com `$shift`

`$shift` acessa o valor de um campo em uma posição diferente da janela — como `LAG`/`LEAD` no SQL.
Útil para detectar variações bruscas entre duas leituras consecutivas.

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
db.temperatura.aggregate([
  { $match: { sensor_id: 'sensor-A1' } },
  { $sort:  { timestamp: 1 } },
  {
    $setWindowFields: {
      partitionBy: '$sensor_id',
      sortBy:      { timestamp: 1 },
      output: {
        leitura_anterior: {
          $shift: { output: '$temperatura_c', by: -1, default: null }
        }
      }
    }
  },
  {
    $addFields: {
      delta: {
        $cond: {
          if:   { $ne: ['$leitura_anterior', null] },
          then: { $subtract: ['$temperatura_c', '$leitura_anterior'] },
          else: null
        }
      }
    }
  },
  // Filtra apenas as variações bruscas (> 1°C em 1 minuto)
  { $match: { $expr: { $gt: [{ $abs: '$delta' }, 1.0] } } },
  { $limit: 8 },
  { $project: { _id: 0, timestamp: 1, temperatura_c: 1, leitura_anterior: 1, delta: 1 } }
])
EOF

### 4.4 Percentis e distribuição estatística

Calcula percentis para entender a distribuição dos dados — útil para definir limiares de alerta
baseados em dados reais ao invés de valores arbitrários. Requer **MongoDB 7.0+**.

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
db.temperatura.aggregate([
  {
    $group: {
      _id:    '$sensor_id',
      p50:    { $percentile: { input: '$temperatura_c', p: [0.50], method: 'approximate' } },
      p90:    { $percentile: { input: '$temperatura_c', p: [0.90], method: 'approximate' } },
      p99:    { $percentile: { input: '$temperatura_c', p: [0.99], method: 'approximate' } },
      desvio: { $stdDevPop: '$temperatura_c' },
      media:  { $avg:       '$temperatura_c' }
    }
  },
  { $sort: { _id: 1 } }
])
EOF

---
## Seção 5 — Caso Real: Pipeline Completo para Sensores IoT

Monitoramento de temperatura em múltiplas salas de um datacenter com 5 sensores.
Objetivo: gerar um **relatório de turno** (3h) com downsampling, alertas e ranking de criticidade.

### 5.1 Criar dataset IoT

5 sensores × leitura a cada 30 segundos × 3 horas = **1.800 documentos**.
A temperatura simula tendência de aquecimento progressivo + onda senoidal + ruído aleatório.

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
db.sensores_iot.drop()
db.createCollection('sensores_iot', {
  timeseries: {
    timeField:   'ts',
    metaField:   'meta',
    granularity: 'seconds'
  }
})

var configs = [
  { id: 'rack-A-top',    sala: 'sala-1', base: 28.0, amp: 3.0 },
  { id: 'rack-A-bottom', sala: 'sala-1', base: 24.0, amp: 2.0 },
  { id: 'rack-B-top',    sala: 'sala-2', base: 30.0, amp: 4.0 },
  { id: 'rack-B-bottom', sala: 'sala-2', base: 25.0, amp: 2.5 },
  { id: 'corredor-frio', sala: 'sala-1', base: 18.0, amp: 1.0 }
];

var docs  = [];
var start = new Date('2024-01-15T08:00:00Z');

for (var seg = 0; seg < 10800; seg += 30) {
  var ts = new Date(start.getTime() + seg * 1000);
  configs.forEach(function(cfg) {
    var tendencia = (seg / 3600) * 1.5;
    var onda      = cfg.amp * Math.sin(2 * Math.PI * seg / 3600);
    var ruido     = (Math.random() - 0.5) * 0.6;
    var temp      = Math.round((cfg.base + tendencia + onda + ruido) * 100) / 100;
    docs.push({
      ts:   ts,
      meta: { sensor_id: cfg.id, sala: cfg.sala },
      temperatura_c: temp,
      cpu_load_pct:  Math.round((20 + Math.random() * 75) * 10) / 10
    });
  });
}

db.sensores_iot.insertMany(docs);
print('Documentos inseridos: ' + docs.length);
EOF

### 5.2 Relatório de turno: temperatura por sala em janelas de 15 minutos

`$dateTrunc` com `binSize: 15` e `unit: 'minute'` divide o turno em blocos de 15 minutos.
Cada bloco por sala recebe um flag de alerta se a temperatura máxima exceder 29°C.

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
db.sensores_iot.aggregate([
  {
    $match: {
      ts: {
        $gte: new Date('2024-01-15T08:00:00Z'),
        $lt:  new Date('2024-01-15T11:00:00Z')
      }
    }
  },
  {
    $group: {
      _id: {
        sala:    '$meta.sala',
        periodo: { $dateTrunc: { date: '$ts', unit: 'minute', binSize: 15 } }
      },
      temp_media: { $avg: '$temperatura_c' },
      temp_max:   { $max: '$temperatura_c' },
      leituras:   { $sum: 1 }
    }
  },
  {
    $addFields: {
      em_alerta: { $gt: ['$temp_max', 29.0] }
    }
  },
  { $sort: { '_id.periodo': 1, '_id.sala': 1 } }
])
EOF

### 5.3 Ranking de sensores por criticidade

Combina temperatura média com quantidade de leituras acima do limiar para gerar um **score de criticidade**,
ordenando os sensores do mais para o menos crítico.

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
db.sensores_iot.aggregate([
  {
    $group: {
      _id:        '$meta.sensor_id',
      sala:       { $first: '$meta.sala' },
      temp_media: { $avg: '$temperatura_c' },
      temp_max:   { $max: '$temperatura_c' },
      qtd_alertas: {
        $sum: { $cond: [{ $gt: ['$temperatura_c', 29.0] }, 1, 0] }
      }
    }
  },
  {
    $addFields: {
      score: { $add: ['$temp_media', { $multiply: ['$qtd_alertas', 0.1] }] }
    }
  },
  { $sort: { score: -1 } },
  { $project: { _id: 0, sensor: '$_id', sala: 1,
                temp_media: 1, temp_max: 1, qtd_alertas: 1, score: 1 } }
])
EOF

### 5.4 Detectar períodos de pico de aquecimento

Usa `window: { range: [-600, 0], unit: 'second' }` para calcular a média dos últimos **10 minutos**
para cada ponto da série, identificando quando o calor acumulado foi mais intenso.

A janela baseada em **tempo** (`range`) é mais precisa que a janela por número de documentos
(`documents`) quando as leituras não têm frequência uniforme.

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
db.sensores_iot.aggregate([
  { $match:  { 'meta.sensor_id': 'rack-B-top' } },
  { $sort:   { ts: 1 } },
  {
    $setWindowFields: {
      partitionBy: '$meta.sensor_id',
      sortBy:      { ts: 1 },
      output: {
        media_10min: {
          $avg:   '$temperatura_c',
          window: { range: [-600, 0], unit: 'second' }
        }
      }
    }
  },
  { $sort:  { media_10min: -1 } },
  { $limit: 5 },
  { $project: { _id: 0, ts: 1, temperatura_c: 1, media_10min: 1 } }
])
EOF

---
## Seção 6 — Performance e Índices

Time Series Collections criam automaticamente um índice composto em `(metaField, timeField)`.
Índices adicionais podem ser criados para campos de metadados frequentemente filtrados.

### 6.1 Inspecionar os índices automáticos

In [ ]:
! mongosh --quiet mongodb://localhost/timeseries_lab --eval 'db.sensores_iot.getIndexes()'

### 6.2 Criar índice secundário em campo de metadado

Acelera filtros em campos de metadados (como `meta.sala`) que não estão cobertos pelo índice automático.

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
db.sensores_iot.createIndex({ 'meta.sala': 1 }, { name: 'idx_sala' })
db.sensores_iot.getIndexes().map(function(i) { return i.name; })
EOF

### 6.3 Analisar o plano de execução com `explain`

- `IXSCAN` → uso de índice — desejável
- `COLLSCAN` → varredura completa — evitar em coleções grandes

Sempre verifique o plano antes de colocar uma query em produção.

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
var plano = db.sensores_iot.find({ 'meta.sala': 'sala-1' }).explain('queryPlanner');
printjson(plano.queryPlanner.winningPlan);
EOF

### 6.4 TTL — Expiração automática de dados antigos

`expireAfterSeconds` instrui o MongoDB a remover automaticamente documentos com `ts`
mais antigo que o limite definido. Elimina a necessidade de jobs de limpeza manuais (cron).

In [ ]:
%%bash
mongosh --quiet mongodb://localhost/timeseries_lab << 'EOF'
// Collection que retém dados por 90 dias
var TTL_DIAS = 90;

db.sensores_retencao_90d.drop()
db.createCollection('sensores_retencao_90d', {
  timeseries: {
    timeField:   'ts',
    metaField:   'meta',
    granularity: 'hours'
  },
  expireAfterSeconds: TTL_DIAS * 24 * 3600
})

var info = db.getCollectionInfos({ name: 'sensores_retencao_90d' })[0];
print('TTL configurado: ' + (info.options.expireAfterSeconds / 86400) + ' dias');
EOF

---
## Resumo

| Conceito | Operador / Método | Quando usar |
|---|---|---|
| Criar coleção TS | `createCollection(..., { timeseries: {...} })` | Sempre que o campo principal for um timestamp |
| Filtro por tempo | `$gte` / `$lt` no `timeField` | Base de qualquer query TS |
| Downsampling | `$dateTrunc` + `$group` | Reduzir resolução para dashboards ou arquivamento |
| Média móvel | `$setWindowFields` + `$avg` + `documents: [-N, 0]` | Suavizar ruídos em séries |
| Soma acumulada | `$setWindowFields` + `$sum` + `unbounded` | Totais progressivos |
| Delta / LAG | `$shift` dentro de `$setWindowFields` | Detectar variações bruscas |
| Janela por tempo | `window: { range: [-N, 0], unit: 'second' }` | Calcular sobre períodos fixos |
| Percentis | `$percentile` dentro de `$group` | Definir limiares baseados em dados reais |
| TTL automático | `expireAfterSeconds` na criação | Política de retenção sem cron jobs |
| Performance | `.explain('queryPlanner')` | Validar uso de índice antes de produção |

### Próximos passos

- Conectar esses dados ao **Grafana** (`3.grafana.ipynb`) para visualização em tempo real
- Usar **Telegraf** (`2.telegraf.ipynb`) para ingerir métricas reais nessas coleções
- Explorar a [documentação oficial de Time Series Collections](https://www.mongodb.com/docs/manual/core/timeseries-collections/)